# Bước 2 — Dịch sách Hán → Việt bằng LLM

**Input:** các file `.txt` của bước 1 (Add Input → Output của notebook OCR).
**Output:** `runs/dich_*/dich/<tên sách>.txt` và `dich_raw/`.

Notebook này chỉ chứa **cấu hình + gọi package**. Toàn bộ logic nằm trong `src/vtm_corpus/`.

**Settings:** Accelerator **GPU T4 x2**, Internet **On**.


## 0. Lấy repo + cài đặt


In [ ]:
# ==== Repo / runtime ====
# Repo công khai: điền URL sau khi publish.
REPO_URL = ""              # vd "https://github.com/you/vtm-corpus.git"
REPO_DIR = "/kaggle/working/vtm-corpus"
INPUT_DIR = "/kaggle/input"   # thư mục chứa PDF (bước 1) hoặc .txt (bước 2)

import subprocess, sys
from pathlib import Path

def sh(cmd, check=True):
    print(">", cmd)
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    if r.stdout.strip(): print(r.stdout.strip())
    if r.stderr.strip(): print(r.stderr.strip())
    if check and r.returncode:
        raise SystemExit(r.returncode)
    return r

local_src = Path.cwd() / "src"
if (local_src / "vtm_corpus").exists():
    REPO_DIR = str(Path.cwd())
    sys.path.insert(0, str(local_src))
elif REPO_URL:
    if not (Path(REPO_DIR) / ".git").exists():
        sh(f"git clone --quiet {REPO_URL} {REPO_DIR}")
    else:
        sh(f"git -C {REPO_DIR} pull --quiet")
    sys.path.insert(0, str(Path(REPO_DIR) / "src"))
else:
    try:
        import vtm_corpus
    except ImportError:
        raise SystemExit("Chưa có repo: điền REPO_URL hoặc chạy notebook trong repo đã clone.")
print("REPO_DIR =", REPO_DIR)


In [ ]:
# Cài package + thư viện model dịch (dùng Python hệ thống của Kaggle).
sh(f'{sys.executable} -m pip install -q -e "{REPO_DIR}[translate]"')
try:  # model cần đăng nhập (Llama, Gemma...): Add-ons -> Secrets -> HF_TOKEN
    from kaggle_secrets import UserSecretsClient
    import os
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    pass
print("Đã cài xong.")


## 1. Xem danh sách đầu vào (PDF hoặc txt)


In [ ]:
# ==== XEM DANH SÁCH TXT ĐẦU VÀO ====
# Chạy ô này trước, rồi điền `books` trong ô cấu hình bên dưới.
#   books=[]                          -> tất cả sách
#   books=["Vệ_sinh_yếu_chỉ_-_tập_4"]  -> đúng 1 sách
#   books=["*ky_huyet*", "*nhi*"]      -> nhiều sách theo glob
from vtm_corpus import list_texts

for i, e in enumerate(list_texts(INPUT_DIR, min_han=50), 1):
    print(f"{i:3}. {e['name']:<50} {e['han']:>7} chữ Hán  {e['path']}")


## 2. Cấu hình (sửa mọi tham số tại đây)


In [ ]:
# ==== CẤU HÌNH DỊCH — sửa mọi trường ngay tại đây ====
from vtm_corpus import TranslateConfig, ReportConfig
from vtm_corpus.config import SYSTEM_PROMPT, USER_PROMPT

DICH = TranslateConfig(
    engine="hf",                      # hf | dummy
    model="Qwen/Qwen2.5-7B-Instruct",
    input_dir=INPUT_DIR,              # thư mục chứa .txt (glob đệ quy)
    books=[],                         # [] = tất cả; tên/glob, vd ["*ky_huyet*"]
    workdir="/kaggle/working",
    run_name="qwen25_7b",             # "" = tên model
    gpu_per_model=2,                  # 1: model <= 4B; 2: model 7-8B
    load_4bit=False,
    batch=8,
    chunk_chars=300,
    max_new_tokens=1200,
    temperature=0.0,
    repetition_penalty=1.05,
    limit=None,                       # vd 20 = chỉ dịch 20 đoạn để thử
    budget_h=10.5,
    stall_min=15,
    max_restarts=5,
    system=SYSTEM_PROMPT,
    user=USER_PROMPT,
    min_han=50,
    device="auto",                    # auto | cpu | cuda
    n_gpu=0,
    worker_python="",                 # "" = sys.executable
    resume_from=[],                   # vd ["/kaggle/input/<notebook-output>"]
    export=True,
    fresh=False,
)

REPORT = ReportConfig(
    preview_pages=3,
    preview_chars=1500,
    layout_images=6,
    show_tables=True,
    image_width=700,
)


### Tải model về trước khi chạy


In [ ]:
# Tải model 1 lần trước khi 2 worker cùng dùng (tránh tải song song).
from huggingface_hub import snapshot_download
snapshot_download(DICH.model, ignore_patterns=["*.gguf", "*.pth", "*.onnx", "original/*"])
print("Đã tải model:", DICH.model)


## 3. Chạy dịch


In [ ]:
# ==== CHẠY DỊCH ====
from vtm_corpus import run_translate

run = run_translate(DICH)
print("run_dir:   ", run.run_dir)
print("output_dir:", run.output_dir)
run


## 4. Xem kết quả ngay trong notebook


In [ ]:
# ==== XEM KẾT QUẢ NGAY TRONG NOTEBOOK ====
from vtm_corpus import show_translate_report, show_translation_pairs

show_translate_report(run.run_dir, REPORT)
show_translation_pairs(run.run_dir, REPORT, n=6)


## 5. Chạy thử nhanh trước khi chạy full


In [ ]:
# ==== CHẠY THỬ NHANH (tuỳ chọn) ====
# DICH.limit = 20
# DICH.engine = "dummy"      # test pipeline không tải model
# run = run_translate(DICH)
# show_translate_report(run.run_dir, REPORT)
